# Categorical Boosting (CatBoost)

Nesse notebook iremos treinar um modelo de CatBoost e iremos comparar seus resultados com os modelos treinados até aqui.

## Importando as Bibliotecas

Primeiro, vamos importar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme()

## Carregando o Dataset

Agora, vamos carregar o dataset.

In [2]:
df = pd.read_parquet("../../data/processed/03_processed.parquet")

In [3]:
df = df.convert_dtypes()

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7919 entries, 0 to 7918
Data columns (total 54 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   tipo_imovel                       7919 non-null   string  
 1   preco                             7919 non-null   Int64   
 2   condominio                        4602 non-null   Int64   
 3   area_m2                           7919 non-null   Int64   
 4   quartos                           7918 non-null   Int64   
 5   banheiros                         7918 non-null   Int64   
 6   vagas                             7286 non-null   Int64   
 7   piscina                           7919 non-null   boolean 
 8   espaco_gourmet                    7919 non-null   boolean 
 9   academia                          7919 non-null   boolean 
 10  spa_massagem                      7919 non-null   boolean 
 11  espaco_lazer                      7919 non-null   boolean 
 12  are

In [5]:
df.head(10)

,tipo_imovel,preco,condominio,area_m2,quartos,banheiros,vagas,piscina,espaco_gourmet,academia,...,distancia_aeroporto_km,faixa_area,faixa_condominio,m2_por_banheiro,m2_por_quarto,banheiros_por_m2,quartos_por_m2,banheiro_por_quarto,quarto_por_vaga,cluster
0,apartamento,350000,720,60,2,2,1,True,True,True,...,16.213545,pequeno,medio,30.0,30.0,0.033333,0.033333,1.0,2.0,4
1,apartamento,511914,<NA>,37,1,1,1,True,True,True,...,7.250204,pequeno,NaN,37.0,37.0,0.027027,0.027027,1.0,1.0,1
2,apartamento,980000,1037,125,3,5,2,True,True,False,...,8.381749,medio,alto,25.0,41.666667,0.04,0.024,1.666667,1.5,1
3,apartamento,2799000,1200,244,3,4,4,True,True,True,...,8.824588,grande,alto,61.0,81.333333,0.016393,0.012295,1.333333,0.75,14
4,apartamento,370000,460,57,2,2,1,True,True,True,...,9.267687,pequeno,medio,28.5,28.5,0.035088,0.035088,1.0,2.0,11
5,apartamento,499000,<NA>,55,2,2,1,True,True,True,...,4.911235,pequeno,NaN,27.5,27.5,0.036364,0.036364,1.0,2.0,7
6,apartamento,195000,350,52,1,1,<NA>,True,True,True,...,9.094186,pequeno,baixo,52.0,52.0,0.019231,0.019231,1.0,<NA>,8
7,apartamento,1210314,<NA>,128,3,3,2,True,True,True,...,6.613218,medio,NaN,42.666667,42.666667,0.023438,0.023438,1.0,1.5,1
8,apartamento,860000,880,105,3,4,2,True,True,True,...,7.747091,medio,alto,26.25,35.0,0.038095,0.028571,1.333333,1.5,1
9,apartamento,400000,400,55,2,2,1,True,True,True,...,3.412207,pequeno,baixo,27.5,27.5,0.036364,0.036364,1.0,2.0,7


## Divisão Treino-Teste

Nessa etapa de modelagem faremos otimização de hiperparâmetros dos nossos modelos e para checarmos a qualidade final do modelo iremos testâ-lo em dados de teste. Para isso precisamos dividir os dados de treinamento e de teste.

In [6]:
from sklearn.model_selection import train_test_split
from utils import random_state

X = df.drop("preco", axis="columns")
y = df["preco"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.8,
    random_state=random_state,
)

/home/gauloish/.dev/ds/projects/meu-ape/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Modelagem

O primeiro modelo que iremos treinar será o nosso baseline, uma regressão linear múltipla. Vamos inicialmente separar as features do nosso dataset baseado nos tipos delas.

In [7]:
numeric_columns = (X_train
    .select_dtypes(include=["number", "boolean"])
    .columns
    .to_list()
)

categorical_columns = (X_train
    .select_dtypes(include=["category", "string"])
    .columns
    .to_list()
)

lower = len(numeric_columns)
upper = lower + len(categorical_columns)

cat_features = list(range(lower, upper))

Agora, vamos montar o nosso pipeline de treinamento para a regressão linear.

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
)
from sklearn.compose import (
    ColumnTransformer,
    TransformedTargetRegressor,
)
from sklearn.pipeline import Pipeline

from feature_engine.transformation import (
    LogCpTransformer,
    YeoJohnsonTransformer,
)

from catboost import CatBoostRegressor

def get_catboost_pipeline(parameters):
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
    )

    model = CatBoostRegressor(
        iterations=parameters["iterations"],
        depth=parameters["depth"],
        learning_rate=parameters["learning_rate"],
        l2_leaf_reg=parameters["l2_leaf_reg"],
        random_strength=parameters["random_strength"],
        bagging_temperature=parameters["bagging_temperature"],
        loss_function="RMSE",
        random_seed=random_state,
        verbose=False,
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return pipeline

def get_catboost_parameters(trial):
    return {
        "iterations": trial.suggest_int("iterations", 500, 2000, step=100),
        "depth": trial.suggest_int("depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 1e2, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 1e1, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 10),
    }

def get_catboost_fit_params():
    return {
        "model__cat_features": cat_features
    }

In [9]:
import utils

study = utils.get_study(
    get_pipeline=get_catboost_pipeline,
    get_parameters=get_catboost_parameters,
    X_train=X_train,
    y_train=y_train,
    n_trials=20,
    fit_params=get_catboost_fit_params()
)

[I 2026-08-24 20:54:11,783] A new study created in memory with name: no-name-313d7f4d-544e-4bee-b564-7861155b5fe5
Best trial: 0. Best value: -725019:   5%|▌         | 1/20 [00:12<04:00, 12.68s/it]

[I 2026-08-24 20:54:24,450] Trial 0 finished with value: -725018.7655171737 and parameters: {'iterations': 1900, 'depth': 2, 'learning_rate': 0.005401732518896231, 'l2_leaf_reg': 0.01021964375098698, 'random_strength': 0.06263052490909785, 'bagging_temperature': 7.271405202969995}. Best is trial 0 with value: -725018.7655171737.


Best trial: 0. Best value: -725019:  10%|█         | 2/20 [00:18<02:37,  8.75s/it]

[I 2026-08-24 20:54:30,454] Trial 1 finished with value: -1187129.0651315039 and parameters: {'iterations': 500, 'depth': 3, 'learning_rate': 0.0012206523306531227, 'l2_leaf_reg': 4.6349551756316885, 'random_strength': 4.254525192165047, 'bagging_temperature': 2.114837475636697}. Best is trial 0 with value: -725018.7655171737.


Best trial: 2. Best value: -602312:  15%|█▌        | 3/20 [00:42<04:23, 15.49s/it]

...

[I 2026-08-24 21:06:19,226] Trial 17 finished with value: -833475.4967271085 and parameters: {'iterations': 900, 'depth': 9, 'learning_rate': 0.0031024823050164263, 'l2_leaf_reg': 25.743903724592446, 'random_strength': 0.14487982451761505, 'bagging_temperature': 7.035955532093736}. Best is trial 5 with value: -600427.2040989883.


Best trial: 5. Best value: -600427:  95%|█████████▌| 19/20 [12:15<00:42, 42.29s/it]

[I 2026-08-24 21:06:27,470] Trial 18 finished with value: -717431.0630553304 and parameters: {'iterations': 700, 'depth': 4, 'learning_rate': 0.05969766201939072, 'l2_leaf_reg': 99.70933748833016, 'random_strength': 0.025038041132320908, 'bagging_temperature': 1.18586625879062}. Best is trial 5 with value: -600427.2040989883.


Best trial: 5. Best value: -600427: 100%|██████████| 20/20 [13:27<00:00, 40.36s/it]

[I 2026-08-24 21:07:39,033] Trial 19 finished with value: -601260.862011751 and parameters: {'iterations': 2000, 'depth': 7, 'learning_rate': 0.09134338903610247, 'l2_leaf_reg': 4.457268205126219, 'random_strength': 1.5142955591302696, 'bagging_temperature': 3.053859515547299}. Best is trial 5 with value: -600427.2040989883.


Vamos olhar as métricas do nosso modelo nos dados de treinamento.

In [10]:
utils.report_scores(
    get_pipeline=get_catboost_pipeline,
    study=study,
    X=X_train,
    y=y_train,
    fit_params=get_catboost_fit_params()
)

R2: 0.84
RMSE: R$ 600427.20
MAE: R$ 249374.73
MedAE: R$ 112841.92
MAPE: 0.23%


E agora nos dados de teste.

In [12]:
utils.report_scores(
    get_pipeline=get_catboost_pipeline,
    study=study,
    X=X_test,
    y=y_test,
    fit_params=get_catboost_fit_params()
)

R2: 0.79
RMSE: R$ 670221.57
MAE: R$ 298957.49
MedAE: R$ 126910.10
MAPE: 0.27%


In [14]:
study.best_params

{'iterations': 1900,
 'depth': 5,
 'learning_rate': 0.023098139161952125,
 'l2_leaf_reg': 6.519721126578009,
 'random_strength': 0.7388778657516146,
 'bagging_temperature': 4.510439888139724}

In [13]:
linear_regression_error_df = utils.get_residual_analysis(
    get_pipeline=get_catboost_pipeline,
    study=study,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=49]="apartamento": Cannot convert 'apartamento' to float